# YOLOv8 Training on Local Machine (Save in Workspace)

In [1]:
import sys
import subprocess

print("Python executable:", sys.executable)

# 🚀 Install PyTorch (nhanh hơn + ổn định hơn)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "torch",
        "torchvision",
        "torchaudio",
        "--index-url",
        "https://download.pytorch.org/whl/cu121",
        "--default-timeout",
        "200",
        "--progress-bar",
        "on"
    ],
    check=True
)

print("\nInstalling Ultralytics and PyYAML...\n")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "ultralytics==8.4.22",
        "PyYAML",
        "--default-timeout",
        "200",
        "-i",
        "https://pypi.tuna.tsinghua.edu.cn/simple"
    ],
    check=True
)

print("\nInstallation complete.")

Python executable: e:\capstone\cleanopsai-ai\myenv\Scripts\python.exe

Installing Ultralytics and PyYAML...


Installation complete.


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
# import sys
# import subprocess

# packages = [
# "torch",
# "torchvision",
# "torchaudio",
# "ultralytics",
# "PyYAML",
# ]

# print("Python executable:", sys.executable)
# print("Uninstalling packages...")

# subprocess.run(
# [sys.executable, "-m", "pip", "uninstall", "-y", *packages],
# check=True
# )

# print("Optional: clearing pip cache...")
# subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], check=False)

# print("Uninstall complete. Restart kernel.")

Python executable: e:\capstone\cleanopsai-ai\.venv\Scripts\python.exe
Uninstalling packages...
Optional: clearing pip cache...
Uninstall complete. Restart kernel.


: 

# Check Local Hardware (GPU/CPU)

In [3]:
import os
import shutil
import subprocess
from pathlib import Path

import ultralytics
import torch

print("Workspace:", Path.cwd())
print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi path:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi, "-L"], check=False)
else:
    print("nvidia-smi not found (this is normal on CPU-only machines).")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("No GPU detected. Notebook will run on CPU.")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\phong\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Workspace: e:\capstone\cleanopsai-ai
Ultralytics version: 8.4.22
PyTorch version: 2.5.1+cu121
Torch CUDA build: 12.1
CUDA available: True
nvidia-smi path: C:\Windows\system32\nvidia-smi.EXE
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU count: 1


In [4]:
from pathlib import Path
import subprocess

dataset_dir = Path.cwd() / "ppe-detection-dataset"
if dataset_dir.exists():
    print(f"Dataset already exists at: {dataset_dir}")
else:
    subprocess.run(
        [
            "git",
            "clone",
            "https://huggingface.co/datasets/jhboyo/ppe-dataset",
            str(dataset_dir),
        ],
        check=True,
    )
    print(f"Dataset cloned to: {dataset_dir}")

Dataset already exists at: e:\capstone\cleanopsai-ai\ppe-detection-dataset


In [5]:
import yaml
from pathlib import Path

candidate_roots = [
    Path.cwd() / "ppe-detection-dataset",
    Path.cwd() / "ppe-dataset",
]

dataset_root = next((p for p in candidate_roots if p.exists()), None)
if dataset_root is None:
    raise FileNotFoundError(
        "Dataset folder not found in workspace. Run the clone cell first, then rerun this cell."
    )

train_dir = dataset_root / "train" / "images"
val_dir_valid = dataset_root / "valid" / "images"
val_dir_val = dataset_root / "val" / "images"

if val_dir_valid.exists():
    val_dir = val_dir_valid
    val_key = "valid/images"
elif val_dir_val.exists():
    val_dir = val_dir_val
    val_key = "val/images"
else:
    raise FileNotFoundError(
        f"Validation images folder not found. Checked: {val_dir_valid} and {val_dir_val}"
    )

if not train_dir.exists():
    raise FileNotFoundError(f"Training images folder not found: {train_dir}")

data = {
    "path": str(dataset_root),
    "train": "train/images",
    "val": val_key,
    "names": {
        0: "helmet",
        1: "vest",
        2: "person",
    },
}

data_yaml = Path.cwd() / "dataset.yaml"
with open(data_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(f"dataset.yaml created at: {data_yaml}")
print(f"Using dataset root: {dataset_root}")
print(f"Using train path: {train_dir}")
print(f"Using val path: {val_dir}")

dataset.yaml created at: e:\capstone\cleanopsai-ai\dataset.yaml
Using dataset root: e:\capstone\cleanopsai-ai\ppe-detection-dataset
Using train path: e:\capstone\cleanopsai-ai\ppe-detection-dataset\train\images
Using val path: e:\capstone\cleanopsai-ai\ppe-detection-dataset\val\images


In [6]:
!yolo settings

JSONDict("C:\Users\phong\AppData\Roaming\Ultralytics\settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "E:\\capstone\\datasets",
  "weights_dir": "E:\\capstone\\cleanopsai-ai\\weights",
  "runs_dir": "E:\\capstone\\cleanopsai-ai\\runs",
  "uuid": "e0cf3cd0678ca48b372ef45433dbc5746bf5f4ccd0c49a7f6a3903e5c7d2a21d",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": false,
  "vscode_msg": true,
  "openvino_msg": true
}
 Learn more about Ultralytics Settings at https://docs.ultralytics.com/quickstart/#ultralytics-settings


In [7]:
import os
import shutil
from pathlib import Path

import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available in PyTorch. Run Cell 2 to install CUDA PyTorch, restart kernel, then rerun checks."
    )

torch.backends.cudnn.benchmark = True
device = 0
imgsz = 640
batch = -1
workers = min(4, os.cpu_count() or 2)
amp = True
train_name = "train_local_gpu"

project_dir = Path.cwd() / "runs"
project_dir.mkdir(parents=True, exist_ok=True)

data_yaml = Path.cwd() / "dataset.yaml"
if not data_yaml.exists():
    raise FileNotFoundError(f"{data_yaml} not found. Run the dataset.yaml cell first.")

print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Training outputs will be saved to: {project_dir}")

model = YOLO("yolov8n.pt")
results = model.train(
    data=str(data_yaml),
    epochs=30,
    imgsz=imgsz,
    device=device,
    batch=batch,
    workers=workers,
    project=str(project_dir),
    name=train_name,
    exist_ok=True,
    cache=True,
    amp=amp,
    save=True,
    plots=True,
    verbose=True,
)

best_src = Path(results.save_dir) / "weights" / "best.pt"
best_dst = Path.cwd() / "best_local_gpu.pt"
if best_src.exists():
    shutil.copy2(best_src, best_dst)
    print(f"Training finished. Save dir: {results.save_dir}")
    print(f"Best model copied to: {best_dst}")
else:
    print(f"Training finished but best.pt not found at: {best_src}")

CUDA device: NVIDIA GeForce RTX 3050 Laptop GPU
Training outputs will be saved to: e:\capstone\cleanopsai-ai\runs
New https://pypi.org/project/ultralytics/8.4.23 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=e:\capstone\cleanopsai-ai\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lr

In [8]:
import glob
import os
from pathlib import Path

base_runs = Path.cwd() / "runs"
models = glob.glob(str(base_runs / "train*" / "weights" / "*.pt"))
models = sorted(models, key=os.path.getmtime, reverse=True)

if not models:
    print(f"No model files found under: {base_runs}")
else:
    print("Latest model files:")
    for m in models[:20]:
        print(m)

best_local = Path.cwd() / "best_local_gpu.pt"
if best_local.exists():
    print(f"Workspace best model: {best_local}")

Latest model files:
e:\capstone\cleanopsai-ai\runs\train_local_gpu\weights\best.pt
e:\capstone\cleanopsai-ai\runs\train_local_gpu\weights\last.pt
Workspace best model: e:\capstone\cleanopsai-ai\best_local_gpu.pt
